# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5C3EE8?style=flat-square&logo=opencv&logoColor=white)](https://opencv.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 4 — Photometric Tracking: Visualización sobre SCARED</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" style="border-radius:50%; object-fit:cover;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### **Objetivos de este notebook**

Replicar la visualización de **Photometric Tracking** del paper Endo-Depth-and-Motion (Recasens et al., 2021) usando los datos SCARED:

1. Cargar el `rgb.mp4` y las poses GT (`frame_data.tar.gz`) de un keyframe de SCARED
2. Predecir el depth map del keyframe con Endo-Depth (igual que en Avance 3)
3. Para cada frame del video, **reproyectar el keyframe** al punto de vista del frame usando la pose GT y el depth map
4. Calcular el **error fotométrico** entre frame actual y keyframe reproyectado
5. Generar la **animación de 5 columnas** replicando el GIF del paper
6. Explorar si la **corrección de iluminación** (Retinex) reduce el error fotométrico

---
## 0. Configuración

### 0.1 Rutas

| Recurso | Ruta en Drive |
|---|---|
| `scared_raw/` | `MyDrive/proyecto_integrador/scared_raw/` |
| `endo_depth_weights/` | `MyDrive/proyecto_integrador/endo_depth_weights/` |
| `Endo-Depth-and-Motion/` | `MyDrive/proyecto_integrador/Endo-Depth-and-Motion/` |
| `avance4_outputs/` | `MyDrive/proyecto_integrador/avance4_outputs/` |

### 0.2 Dataset seleccionado

Usamos `dataset_1 / keyframe_1` porque es el primer dataset disponible y el keyframe con más frames de video (197 frames a 25 FPS ≈ 8 segundos de movimiento real del endoscopio).

In [ ]:
# Instalación de dependencias (solo necesario en Colab)
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import torch
    print(f"PyTorch ya instalado: {torch.__version__}")
except ImportError:
    pip_install("torch torchvision --index-url https://download.pytorch.org/whl/cu118")

try:
    import tifffile
except ImportError:
    pip_install("tifffile")

try:
    from skimage import __version__
except ImportError:
    pip_install("scikit-image")

import torch, tifffile, cv2, numpy as np
print(f"torch    : {torch.__version__}")
print(f"tifffile : {tifffile.__version__}")
print(f"opencv   : {cv2.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE        = Path("/content/drive/MyDrive/proyecto_integrador")
    MODEL_PATH  = BASE / "endo_depth_weights"
    SCARED_ROOT = BASE / "scared_raw"
    EDAM_PATH   = BASE / "Endo-Depth-and-Motion"
    OUT_DIR     = BASE / "avance4_outputs"
else:
    MODEL_PATH  = Path("E:/endo_depth_weights")
    SCARED_ROOT = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH   = Path("E:/Endo-Depth-and-Motion")
    OUT_DIR     = Path("../outcomes/avance4")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset a analizar
DATASET_ID  = "dataset_1"
KEYFRAME_ID = "keyframe_1"

# Submuestreo de frames para la animación (1 = todos, 4 = cada 4to)
FRAME_STEP = 4

print(f"Entorno  : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Dataset  : {DATASET_ID} / {KEYFRAME_ID}")
print(f"OUT_DIR  : {OUT_DIR}")

---
## 1. Cargar Endo-Depth

Mismo modelo que en Avance 3: ResNet18 encoder + DepthDecoder, pesos de Hamlyn.

In [ ]:
import sys
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

encoder = ResnetEncoder(18, False)
loaded_enc = torch.load(MODEL_PATH / "encoder.pth", map_location=DEVICE)
FEED_HEIGHT = loaded_enc["height"]
FEED_WIDTH  = loaded_enc["width"]

filtered_enc = {k: v for k, v in loaded_enc.items() if k in encoder.state_dict()}
encoder.load_state_dict(filtered_enc)
encoder.to(DEVICE).eval()

depth_decoder = DepthDecoder(num_ch_enc=encoder.num_ch_enc, scales=range(4))
loaded_dec = torch.load(MODEL_PATH / "depth.pth", map_location=DEVICE)
depth_decoder.load_state_dict(loaded_dec)
depth_decoder.to(DEVICE).eval()

print(f"Resolución del modelo: {FEED_HEIGHT}×{FEED_WIDTH}")
print("Modelo cargado ✓")

---
## 2. Cargar keyframe, video y poses

### Estructura del `rgb.mp4`

El video es **estéreo apilado verticalmente**: resolución 1280×2048, donde:
- Mitad superior (filas 0–1023): cámara **izquierda** (la misma que `Left_Image.png`)
- Mitad inferior (filas 1024–2047): cámara **derecha**

Usamos solo el canal izquierdo para ser consistentes con el Avance 3.

### Estructura de `frame_data.tar.gz`

197 archivos JSON (`frame_data000000.json` … `frame_data000196.json`), uno por frame del video. Cada JSON contiene:
- `camera-pose`: matriz **4×4 homogénea** con la pose de la cámara en el frame (R|t en la última columna, escala en metros)
- `camera-calibration.KL`: matriz intrínseca 3×3 de la cámara izquierda

In [ ]:
import zipfile, tarfile, io, json, cv2, tifffile
import numpy as np

zip_path = SCARED_ROOT / f"{DATASET_ID}.zip"

def load_from_zip(zip_path, inner_path):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(inner_path) as f:
            return f.read()

# ── Keyframe RGB ──────────────────────────────────────────────────────────────
kf_prefix = f"{DATASET_ID}/{KEYFRAME_ID}"
img_bytes = load_from_zip(zip_path, f"{kf_prefix}/Left_Image.png")
buf = np.frombuffer(img_bytes, np.uint8)
keyframe_rgb = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

# ── Keyframe depth GT ────────────────────────────────────────────────────────
tiff_bytes = load_from_zip(zip_path, f"{kf_prefix}/left_depth_map.tiff")
tiff = tifffile.imread(io.BytesIO(tiff_bytes))
depth_gt = tiff[..., 2].astype(np.float32)   # canal Z en mm
depth_gt[depth_gt <= 0] = np.nan

# ── Video frames (canal izquierdo) ───────────────────────────────────────────
video_bytes = load_from_zip(zip_path, f"{kf_prefix}/data/rgb.mp4")
tmp_mp4 = OUT_DIR / "_tmp_video.mp4"
tmp_mp4.write_bytes(video_bytes)

cap = cv2.VideoCapture(str(tmp_mp4))
all_frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    h = frame.shape[0] // 2
    left = cv2.cvtColor(frame[:h], cv2.COLOR_BGR2RGB)
    all_frames.append(left)
cap.release()
tmp_mp4.unlink()

# ── Poses GT ─────────────────────────────────────────────────────────────────
fd_bytes = load_from_zip(zip_path, f"{kf_prefix}/data/frame_data.tar.gz")
with tarfile.open(fileobj=io.BytesIO(fd_bytes)) as t:
    poses = []
    K = None
    for i in range(len(all_frames)):
        name = f"frame_data{i:06d}.json"
        d = json.load(t.extractfile(name))
        poses.append(np.array(d["camera-pose"], dtype=np.float64))
        if K is None:
            K = np.array(d["camera-calibration"]["KL"], dtype=np.float64)

print(f"Keyframe    : {keyframe_rgb.shape}  dtype={keyframe_rgb.dtype}")
print(f"Depth GT    : {depth_gt.shape}  rango=[{np.nanmin(depth_gt):.1f}, {np.nanmax(depth_gt):.1f}] mm")
print(f"Video frames: {len(all_frames)}  resolución={all_frames[0].shape}")
print(f"Poses GT    : {len(poses)} matrices 4×4")
print(f"K (left)    :\n{K}")

---
## 3. Predecir depth map del keyframe con Endo-Depth

Usamos el mismo pipeline de inferencia del Avance 3. Aplicamos median scaling con el GT disponible para convertir a mm.

In [ ]:
import torch.nn.functional as F
import PIL.Image as pil
from torchvision import transforms
import matplotlib.pyplot as plt

def predict_depth(img_rgb, encoder, decoder, feed_h, feed_w, device):
    H, W = img_rgb.shape[:2]
    input_pil = pil.fromarray(img_rgb).resize((feed_w, feed_h), pil.LANCZOS)
    input_t = transforms.ToTensor()(input_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        features = encoder(input_t)
        outputs  = decoder(features)
    disp = outputs[("disp", 0)]
    disp_full = F.interpolate(disp, (H, W), mode="bilinear", align_corners=False)
    disp_np = disp_full.squeeze().cpu().numpy()
    min_disp, max_disp = 1.0/100.0, 1.0/0.1
    scaled = min_disp + (max_disp - min_disp) * disp_np
    return 1.0 / scaled

depth_rel = predict_depth(
    keyframe_rgb, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE)

# Median scaling con GT
CAP_MM = 150.0
valid = (~np.isnan(depth_gt)) & (depth_gt > 0) & (depth_gt < CAP_MM)
scale = np.median(depth_gt[valid]) / (np.median(depth_rel[valid]) + 1e-8)
depth_pred_mm = depth_rel * scale

print(f"Scale factor    : {scale:.4f}")
print(f"Depth pred rango: [{depth_pred_mm.min():.1f}, {depth_pred_mm.max():.1f}] mm")

# Visualización diagnóstica
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(keyframe_rgb)
axes[0].set_title("Keyframe (Left_Image.png)", fontsize=11)
axes[0].axis("off")

im = axes[1].imshow(depth_pred_mm, cmap="jet", vmin=0,
                    vmax=np.nanpercentile(depth_pred_mm, 98))
axes[1].set_title("Depth Endo-Depth (mm)", fontsize=11)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], label="mm")

im2 = axes[2].imshow(depth_gt, cmap="jet", vmin=0,
                     vmax=np.nanpercentile(depth_gt, 98))
axes[2].set_title("Depth GT (luz estructurada)", fontsize=11)
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], label="mm")

plt.suptitle(f"{DATASET_ID}/{KEYFRAME_ID} — Comparativa depth", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_depth_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Reproyección fotométrica

### Teoría

Dado el keyframe $I_k$ con depth map $D_k$, y la pose relativa entre el keyframe y el frame actual $T_{k \to t}$, podemos proyectar cada píxel del keyframe al frame actual:

$$p_t = K \cdot T_{k \to t} \cdot D_k(p_k) \cdot K^{-1} \cdot p_k$$

donde:
- $p_k$ = coordenada homogénea del píxel en el keyframe
- $D_k(p_k)$ = profundidad de ese píxel en mm
- $T_{k \to t}$ = pose relativa (4×4): cómo moverse del keyframe al frame $t$
- $K$ = matriz intrínseca
- $p_t$ = dónde aparece ese píxel en el frame $t$

El **error fotométrico** es $|I_t(p_t) - I_k(p_k)|$: qué tan diferente es el color del frame actual vs. el color reproyectado del keyframe. Si el depth map y la pose son perfectos, este error debería ser cero.

### Pose relativa

Las poses en `frame_data` son absolutas $T_{world \to cam}$. La pose relativa del keyframe al frame $t$ es:

$$T_{k \to t} = T_t \cdot T_k^{-1}$$

In [ ]:
import cv2
import numpy as np

# Pose del keyframe = pose del frame 0 del video
T_keyframe = poses[0]   # 4×4

def reproject_keyframe(keyframe_rgb, depth_mm, T_current, T_ref, K):
    """
    Reproyecta el keyframe al punto de vista del frame actual.

    Parámetros
    ----------
    keyframe_rgb : np.ndarray (H, W, 3) uint8
    depth_mm     : np.ndarray (H, W) float — depth del keyframe en mm
    T_current    : np.ndarray (4, 4) — pose absoluta del frame actual
    T_ref        : np.ndarray (4, 4) — pose absoluta del keyframe
    K            : np.ndarray (3, 3) — intrínsecos

    Retorna
    -------
    reprojected  : np.ndarray (H, W, 3) uint8 — keyframe visto desde T_current
    mask_valid   : np.ndarray (H, W) bool — True donde hay reproyección válida
    """
    H, W = depth_mm.shape
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    # Grid de coordenadas de píxel del keyframe
    u = np.arange(W, dtype=np.float64)
    v = np.arange(H, dtype=np.float64)
    uu, vv = np.meshgrid(u, v)

    Z = depth_mm.astype(np.float64)
    # Profundidad en metros para la transformación de pose
    # (las poses de SCARED están en metros, depth_mm en mm → convertir)
    Z_m = Z / 1000.0

    # Back-project: píxeles del keyframe → puntos 3D en ref del keyframe
    X = (uu - cx) * Z_m / fx
    Y = (vv - cy) * Z_m / fy

    # Stack en (4, H*W) — coordenadas homogéneas
    pts_kf = np.stack([X, Y, Z_m, np.ones_like(Z_m)], axis=0).reshape(4, -1)

    # Pose relativa: T_current * inv(T_ref)
    T_rel = T_current @ np.linalg.inv(T_ref)   # (4, 4)

    # Transformar al sistema de coordenadas del frame actual
    pts_curr = T_rel @ pts_kf   # (4, H*W)

    # Project: 3D → píxeles del frame actual
    Xc, Yc, Zc = pts_curr[0], pts_curr[1], pts_curr[2]
    mask_valid = Zc > 1e-3   # solo puntos delante de la cámara

    u_proj = np.where(mask_valid, fx * Xc / (Zc + 1e-8) + cx, 0)
    v_proj = np.where(mask_valid, fy * Yc / (Zc + 1e-8) + cy, 0)

    # También ignorar píxeles fuera del campo de la imagen
    mask_valid &= (u_proj >= 0) & (u_proj < W - 1) & \
                  (v_proj >= 0) & (v_proj < H - 1)

    # Mapa de flujo para cv2.remap
    map_x = u_proj.reshape(H, W).astype(np.float32)
    map_y = v_proj.reshape(H, W).astype(np.float32)
    mask_2d = mask_valid.reshape(H, W)

    reprojected = cv2.remap(
        keyframe_rgb, map_x, map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0
    )
    reprojected[~mask_2d] = 0

    return reprojected, mask_2d


# Prueba rápida con el frame 10
test_frame_idx = 10
reproj, mask = reproject_keyframe(
    keyframe_rgb, depth_pred_mm,
    poses[test_frame_idx], T_keyframe, K
)

actual = all_frames[test_frame_idx]
error = np.abs(actual.astype(np.float32) - reproj.astype(np.float32)).mean(axis=2)
error[~mask] = np.nan

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img, title in zip(axes,
    [keyframe_rgb, depth_pred_mm, actual, reproj, error],
    ["Keyframe", "Depth", "Actual", "Reprojected", "Error"]):
    if title == "Depth":
        ax.imshow(img, cmap="jet", vmin=0, vmax=np.nanpercentile(depth_pred_mm, 98))
    elif title == "Error":
        ax.imshow(img, cmap="hot", vmin=0, vmax=50)
    elif title in ("Actual", "Reprojected"):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cmap="gray")
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis("off")

plt.suptitle(f"Reproyección fotométrica — frame {test_frame_idx}",
             fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_reproject_test.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Error fotométrico medio (px válidos): {np.nanmean(error):.2f}")

---
## 5. Animación completa — replicando el GIF de Endo-Depth-and-Motion

Generamos un frame por cada `FRAME_STEP` frames del video, con las 5 columnas del paper:
**Keyframe | Depth | Actual | Reprojected | Error**

El resultado se guarda como GIF y como MP4.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from IPython.display import HTML, display

frame_indices = list(range(0, len(all_frames), FRAME_STEP))
print(f"Generando animación: {len(frame_indices)} frames (de {len(all_frames)} totales)")

depth_vmax = float(np.nanpercentile(depth_pred_mm, 98))

# Pre-calcular todas las reproyecciones
reproj_frames = []
error_frames  = []
mean_errors   = []

for idx in frame_indices:
    reproj, mask = reproject_keyframe(
        keyframe_rgb, depth_pred_mm, poses[idx], T_keyframe, K)
    actual = all_frames[idx]
    err = np.abs(actual.astype(np.float32) - reproj.astype(np.float32)).mean(axis=2)
    err[~mask] = np.nan
    reproj_frames.append(reproj)
    error_frames.append(err)
    mean_errors.append(float(np.nanmean(err)))

print(f"Error fotométrico medio — min: {min(mean_errors):.2f}  max: {max(mean_errors):.2f}")

# ── Construir animación ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.patch.set_facecolor("black")
for ax in axes:
    ax.set_facecolor("black")
    ax.axis("off")

col_labels = ["Keyframe", "Depth", "Actual", "Reprojected", "Error"]
for ax, lbl in zip(axes, col_labels):
    ax.set_title(lbl, color="white", fontsize=12, fontweight="bold", pad=4)

# Imágenes iniciales
kf_gray = cv2.cvtColor(keyframe_rgb, cv2.COLOR_RGB2GRAY)
im0 = axes[0].imshow(keyframe_rgb)
im1 = axes[1].imshow(depth_pred_mm, cmap="jet", vmin=0, vmax=depth_vmax)
im2 = axes[2].imshow(cv2.cvtColor(all_frames[0], cv2.COLOR_RGB2GRAY), cmap="gray",
                     vmin=0, vmax=255)
im3 = axes[3].imshow(cv2.cvtColor(reproj_frames[0], cv2.COLOR_RGB2GRAY), cmap="gray",
                     vmin=0, vmax=255)
im4 = axes[4].imshow(error_frames[0], cmap="hot", vmin=0, vmax=50)

title_txt = fig.suptitle("", color="white", fontsize=11)
plt.tight_layout()

def update(i):
    actual_gray = cv2.cvtColor(all_frames[frame_indices[i]], cv2.COLOR_RGB2GRAY)
    reproj_gray = cv2.cvtColor(reproj_frames[i], cv2.COLOR_RGB2GRAY)
    im2.set_data(actual_gray)
    im3.set_data(reproj_gray)
    im4.set_data(error_frames[i])
    title_txt.set_text(
        f"{DATASET_ID}/{KEYFRAME_ID}  |  frame {frame_indices[i]:03d}/{len(all_frames)}  "
        f"|  error fotométrico medio: {mean_errors[i]:.1f} px")
    return im2, im3, im4, title_txt

ani = animation.FuncAnimation(
    fig, update, frames=len(frame_indices), interval=100, blit=False)

# Guardar GIF
gif_path = OUT_DIR / "avance4_photometric_tracking.gif"
ani.save(str(gif_path), writer="pillow", fps=10, dpi=80)
print(f"GIF guardado: {gif_path}")

# Mostrar en notebook
plt.close()
display(HTML(ani.to_jshtml()))

---
## 6. Error fotométrico a lo largo del video

El error fotométrico medio por frame nos dice cómo varía la calidad de la reproyección conforme el endoscopio se aleja del keyframe. Esperamos que aumente monotónicamente: a mayor desplazamiento, mayor error.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(frame_indices, mean_errors, color="#E0A800", linewidth=2)
axes[0].set_xlabel("Frame del video")
axes[0].set_ylabel("Error fotométrico medio (DN)")
axes[0].set_title("Error fotométrico vs. posición en el video", fontsize=12)
axes[0].grid(alpha=0.3)

# Desplazamiento de la cámara respecto al keyframe
displacements = []
for idx in frame_indices:
    T_rel = poses[idx] @ np.linalg.inv(T_keyframe)
    d = np.linalg.norm(T_rel[:3, 3]) * 1000   # en mm
    displacements.append(d)

axes[1].scatter(displacements, mean_errors, c=frame_indices,
                cmap="viridis", alpha=0.7, s=20)
axes[1].set_xlabel("Desplazamiento de cámara desde keyframe (mm)")
axes[1].set_ylabel("Error fotométrico medio (DN)")
axes[1].set_title("Error vs. desplazamiento espacial", fontsize=12)
axes[1].grid(alpha=0.3)

# Correlación
r = np.corrcoef(displacements, mean_errors)[0, 1]
axes[1].text(0.05, 0.92, f"r = {r:.3f}",
             transform=axes[1].transAxes, fontsize=11)

plt.suptitle(f"Análisis del error fotométrico — {DATASET_ID}/{KEYFRAME_ID}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_error_fotometrico.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Desplazamiento máximo  : {max(displacements):.1f} mm")
print(f"Error mín / máx        : {min(mean_errors):.2f} / {max(mean_errors):.2f} DN")
print(f"Correlación r(dist, err): {r:.3f}")

---
## 7. ¿Retinex reduce el error fotométrico?

Hipótesis de este avance: si aplicamos Retinex al keyframe antes de predecir el depth map y antes de calcular el error fotométrico, ¿el error baja? Esto extendería el resultado del Avance 3 (Retinex mejora AbsRel vs. GT) a un escenario de tracking en movimiento.

Comparamos dos condiciones:
- **none**: keyframe original → depth Endo-Depth → reproyectar
- **retinex**: keyframe corregido → depth Endo-Depth → reproyectar

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def correct_retinex(img_rgb, sigma=30):
    img_f = img_rgb.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:, :, c], (0, 0), sigma)
        result[:, :, c] = np.log(img_f[:, :, c]) - np.log(blur + 1.0)
    result -= result.min()
    return (result / (result.max() + 1e-8) * 255).astype(np.uint8)

# Depth con Retinex
keyframe_retinex = correct_retinex(keyframe_rgb)
depth_retinex_rel = predict_depth(
    keyframe_retinex, encoder, depth_decoder, FEED_HEIGHT, FEED_WIDTH, DEVICE)
scale_r = np.median(depth_gt[valid]) / (np.median(depth_retinex_rel[valid]) + 1e-8)
depth_retinex_mm = depth_retinex_rel * scale_r

# Calcular errores para ambas condiciones sobre todos los frames
errors_none    = []
errors_retinex = []

for idx in frame_indices:
    actual = all_frames[idx]

    # none
    reproj_n, mask_n = reproject_keyframe(
        keyframe_rgb, depth_pred_mm, poses[idx], T_keyframe, K)
    err_n = np.abs(actual.astype(np.float32) - reproj_n.astype(np.float32)).mean(axis=2)
    err_n[~mask_n] = np.nan
    errors_none.append(float(np.nanmean(err_n)))

    # retinex
    reproj_r, mask_r = reproject_keyframe(
        keyframe_retinex, depth_retinex_mm, poses[idx], T_keyframe, K)
    err_r = np.abs(actual.astype(np.float32) - reproj_r.astype(np.float32)).mean(axis=2)
    err_r[~mask_r] = np.nan
    errors_retinex.append(float(np.nanmean(err_r)))

# Comparativa
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(frame_indices, errors_none,    label="Sin corrección (none)",
        color="#2766CB", linewidth=2)
ax.plot(frame_indices, errors_retinex, label="Retinex SSR (σ=30)",
        color="#E0A800", linewidth=2, linestyle="--")
ax.set_xlabel("Frame del video")
ax.set_ylabel("Error fotométrico medio (DN)")
ax.set_title("Impacto de Retinex sobre el error fotométrico durante el tracking",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

mean_none    = np.nanmean(errors_none)
mean_retinex = np.nanmean(errors_retinex)
mejora = (mean_none - mean_retinex) / mean_none * 100
ax.text(0.02, 0.90,
        f"Media none={mean_none:.2f}  retinex={mean_retinex:.2f}  "
        f"mejora={mejora:+.1f}%",
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.savefig(OUT_DIR / "avance4_retinex_vs_none.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Error medio none    : {mean_none:.2f} DN")
print(f"Error medio retinex : {mean_retinex:.2f} DN")
print(f"Diferencia          : {mejora:+.1f}%")

---
## 8. Conclusiones

### Lo que replicamos

La visualización de **Photometric Tracking** del paper Endo-Depth-and-Motion: keyframe estático con depth map predicho, reproyectado fotométricamente a lo largo de la secuencia de video del `rgb.mp4`, usando las poses GT de `frame_data.tar.gz` como referencia.

### Diferencia con el paper

En el paper, las poses se **estiman** fotométricamente (el tracker las optimiza sin GT). Aquí usamos directamente las poses GT — lo que nos permite aislar el efecto de la calidad del depth map sobre el error de reproyección, sin confundirlo con errores de estimación de pose.

### Implicaciones para el proyecto

El error fotométrico durante el tracking es una métrica complementaria al AbsRel del Avance 3: mide si la mejora de profundidad se traduce en mejor rastreo visual a lo largo de un clip real. Si Retinex reduce este error, la hipótesis del proyecto se confirma también en el escenario dinámico más relevante clínicamente.

## Referencias

- Recasens, D., et al. (2021). Endo-Depth-and-Motion: Reconstruction and Tracking in Endoscopic Videos Using Depth Networks and Photometric Constraints. *IEEE RA-L*, 6(4), 7225–7232.
- Allan, M., et al. (2021). Stereo Correspondence and Reconstruction of Endoscopic Data Challenge. *arXiv:2101.01133*.